## 基本的なRAGシステムの構築
- `CondensePlusContextChatEngine` を利用して試す。



In [1]:
import os
from dotenv import load_dotenv
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.openai import OpenAI

from llama_index.core.chat_engine import CondensePlusContextChatEngine

# 環境変数の取得
load_dotenv("../.env")
os.environ['OPENAI_API_KEY']  = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# ドキュメント読み込み＆インデックス
documents = SimpleDirectoryReader("./data/text").load_data()
index = VectorStoreIndex.from_documents(documents)


2026-02-25 09:46:56,642 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [3]:
# Retriever（top_k 相当）
retriever = index.as_retriever(similarity_top_k=3)

# LLM
llm = OpenAI(model=MODEL_NAME)

# Chat Engine（condense + context）
chat_engine = CondensePlusContextChatEngine.from_defaults(
    retriever=retriever,
    llm=llm,
    # ここに template を渡せる引数がある場合は from_defaults のシグネチャに合わせて追加
    verbose=True,
)


In [4]:
# 質問：1回目
response = chat_engine.chat("有給休暇はいつから取得できますか？")

# 言語モデルからの回答を表示
print(response)


2026-02-25 09:48:34,140 - INFO - Condensed question: 有給休暇はいつから取得できますか？


Condensed question: 有給休暇はいつから取得できますか？


2026-02-25 09:48:36,732 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-25 09:48:39,670 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


有給休暇は、入社から6ヶ月継続勤務し、全労働日の8割以上出勤した場合に初めて付与されます。初年度には10日間の有給休暇が付与され、その後は勤続年数に応じて増加します。ですので、実際に有給休暇を取得できるのは、入社から6ヶ月後になります。


In [6]:
# 引用元を表示
for sn in response.source_nodes:
    print("ファイル名：", sn.node.metadata.get("file_name"))
    print("関連度スコア:", sn.score)
    print("テキスト：")
    print(sn.node.text)
    print("-" * 50)


ファイル名： 03休暇規則.md
関連度スコア: 0.8744311031347539
テキスト：
**産前産後休暇**

   - **産前休暇**：出産予定日の**6週間前**から取得可能です。
   - **産後休暇**：出産日の翌日から**8週間**は就業が禁止されています。
   - 産前産後休暇中は、健康保険から出産手当金が支給されます。

3. **育児休業**

   - 子供が**1歳**になるまでの間、育児休業を取得できます。
   - 保育所に入れないなどの事情がある場合、最長で**2歳**まで延長可能です。
   - 育児休業中は、雇用保険から育児休業給付金が支給されます。

4. **介護休業**

   - 要介護状態にある家族を介護するために、**通算93日間**の介護休業を取得できます。
   - 介護休業は、対象家族一人につき1回、分割して最大3回まで取得可能です。

5. **生理休暇**

   - 女性従業員で、生理により就業が困難な場合は、申請により休暇を取得できます。

6. **裁判員休暇**

   - 裁判員や補充裁判員として選任された場合、その期間中は休暇を取得できます。

### 3. 特別有給休暇

会社が特別に認めた有給の休暇です。

1. **リフレッシュ休暇**

   - **勤続5年**ごとに、連続した**5日間**のリフレッシュ休暇が取得できます。
   - リフレッシュ休暇は、有給休暇とは別に付与されます。

2. **ボランティア休暇**

   - 社会貢献活動を支援するため、年間**2日間**のボランティア休暇を取得できます。
   - ボランティア休暇を取得する際は、活動内容を事前に上司へ報告してください。

### 4. 無給休暇

給与の支給がない休暇です。

1. **自己啓発休業**

   - 自己啓発や留学などの目的で、最長**2年間**の休業が可能です。
   - 休業期間中は、社会保険料の自己負担などが発生します。

2. **私傷病休業**

   - 病気やけがで長期間の治療が必要な場合、最長**1年間**の休業が可能です。
   - 休業中は、健康保険から傷病手当金が支給される場合があります。

### 5. 休暇取得の手続き

1.
----------------

In [7]:
# 質問：2回目
response = chat_engine.chat("勤続年数が5年の場合は何日ですか？")

# 言語モデルからの回答を表示
print(response)


2026-02-25 09:57:15,993 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-25 09:57:15,997 - INFO - Condensed question: 勤続年数が5年の場合、有給休暇は何日付与されますか？
2026-02-25 09:57:16,189 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Condensed question: 勤続年数が5年の場合、有給休暇は何日付与されますか？


2026-02-25 09:57:18,514 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


勤続年数が5年の場合、年次有給休暇は18日間付与されます。これは、勤続年数に応じて増加する有給休暇の日数の一部です。


In [8]:
# 引用元を表示
for sn in response.source_nodes:
    print("ファイル名：", sn.node.metadata.get("file_name"))
    print("関連度スコア:", sn.score)
    print("テキスト：")
    print(sn.node.text)
    print("-" * 50)

ファイル名： 03休暇規則.md
関連度スコア: 0.858029492091321
テキスト：
**産前産後休暇**

   - **産前休暇**：出産予定日の**6週間前**から取得可能です。
   - **産後休暇**：出産日の翌日から**8週間**は就業が禁止されています。
   - 産前産後休暇中は、健康保険から出産手当金が支給されます。

3. **育児休業**

   - 子供が**1歳**になるまでの間、育児休業を取得できます。
   - 保育所に入れないなどの事情がある場合、最長で**2歳**まで延長可能です。
   - 育児休業中は、雇用保険から育児休業給付金が支給されます。

4. **介護休業**

   - 要介護状態にある家族を介護するために、**通算93日間**の介護休業を取得できます。
   - 介護休業は、対象家族一人につき1回、分割して最大3回まで取得可能です。

5. **生理休暇**

   - 女性従業員で、生理により就業が困難な場合は、申請により休暇を取得できます。

6. **裁判員休暇**

   - 裁判員や補充裁判員として選任された場合、その期間中は休暇を取得できます。

### 3. 特別有給休暇

会社が特別に認めた有給の休暇です。

1. **リフレッシュ休暇**

   - **勤続5年**ごとに、連続した**5日間**のリフレッシュ休暇が取得できます。
   - リフレッシュ休暇は、有給休暇とは別に付与されます。

2. **ボランティア休暇**

   - 社会貢献活動を支援するため、年間**2日間**のボランティア休暇を取得できます。
   - ボランティア休暇を取得する際は、活動内容を事前に上司へ報告してください。

### 4. 無給休暇

給与の支給がない休暇です。

1. **自己啓発休業**

   - 自己啓発や留学などの目的で、最長**2年間**の休業が可能です。
   - 休業期間中は、社会保険料の自己負担などが発生します。

2. **私傷病休業**

   - 病気やけがで長期間の治療が必要な場合、最長**1年間**の休業が可能です。
   - 休業中は、健康保険から傷病手当金が支給される場合があります。

### 5. 休暇取得の手続き

1.
-----------------